In [1]:
!pip install torch transformers


In [2]:
import torch
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [3]:
FULL_MENU = [
    # Burgers
    {"name": "Beef Burger", "category": "Burger", "price": 150, "desc": "Classic grilled beef patty with lettuce and sauce."},
    {"name": "Beef Burger (Cheese)", "category": "Burger", "price": 180, "desc": "Grilled beef patty topped with melted cheddar cheese."},
    {"name": "Beef Burger (Jumbo Cheese)", "category": "Burger", "price": 250, "desc": "Massive double beef patty with two layers of melted cheese."},
    {"name": "Chicken Burger", "category": "Burger", "price": 140, "desc": "Tender grilled chicken breast with lettuce and mayo."},
    {"name": "Chicken Burger (Cheese)", "category": "Burger", "price": 170, "desc": "Grilled chicken topped with melted cheese and garlic sauce."},
    {"name": "Zinger Burger", "category": "Burger", "price": 160, "desc": "Crispy fried spicy chicken fillet with creamy pepper mayo."},
    {"name": "Zinger Burger (Cheese)", "category": "Burger", "price": 190, "desc": "Crispy spicy chicken with melted cheese and sauce."},

    # Rolls
    {"name": "Beef Roll", "category": "Roll", "price": 100, "desc": "Spicy grilled beef wrapped in soft flatbread with onions."},
    {"name": "Beef Cheese Roll", "category": "Roll", "price": 120, "desc": "Spicy beef wrap with melted cheese and tangy sauce."},
    {"name": "Chicken Mayo Roll", "category": "Roll", "price": 110, "desc": "Grilled chicken wrapped with creamy mayonnaise and lettuce."},
    {"name": "Chicken Cheese Roll", "category": "Roll", "price": 130, "desc": "Grilled chicken with melted cheese in a soft wrap."},
    {"name": "Kabab Roll", "category": "Roll", "price": 90, "desc": "Spicy minced meat kabab wrapped with mint chutney."},
    {"name": "Kabab Cheese Jumbo Roll", "category": "Roll", "price": 200, "desc": "Massive jumbo roll with double meat and double cheese."},

    # BBQ
    {"name": "Chicken Bihari Tikka (Leg)", "category": "BBQ", "price": 200, "desc": "Spicy Bihari-marinated grilled chicken leg."},
    {"name": "Chicken Bihari Tikka (Chest)", "category": "BBQ", "price": 220, "desc": "Large chicken breast marinated in Bihari spices, chargrilled."},
    {"name": "Malai Boti", "category": "BBQ", "price": 180, "desc": "Creamy yogurt-marinated chicken boti, grilled until golden."},
    {"name": "Seekh Kabab", "category": "BBQ", "price": 160, "desc": "Spicy minced beef rolls grilled on skewers."},
    {"name": "Reshmi Kabab", "category": "BBQ", "price": 170, "desc": "Silky, melt-in-your-mouth minced chicken patties."},

    # Pizza
    {"name": "Pizza Large", "category": "Pizza", "price": 450, "desc": "Large 14-inch pizza with cheese and tomato sauce."},
    {"name": "Pizza Regular", "category": "Pizza", "price": 350, "desc": "Medium pizza with soft crust and gooey cheese."},
    {"name": "Supper Suprem", "category": "Pizza", "price": 550, "desc": "Ultimate loaded pizza with meat, veggies, and extra cheese."},
    {"name": "Extra Cheese Topping", "category": "Pizza", "price": 100, "desc": "Additional layer of stretchy mozzarella cheese."},

    # Pasta
    {"name": "Passta Large", "category": "Pasta", "price": 300, "desc": "Large creamy penne pasta in tomato and cheese sauce."},
    {"name": "Passta Small", "category": "Pasta", "price": 200, "desc": "Small portion of creamy penne pasta with herbs."},
]

CATEGORY_DESCRIPTIONS = {
    "Burger": "A sandwich with a grilled or fried patty (beef or chicken) served in a bun with lettuce, tomatoes, and sauce.",
    "Roll": "A soft flatbread or tortilla wrap stuffed with meat, vegetables, and sauce.",
    "BBQ": "Traditional grilled meats like tikka, kababs, and boti cooked over a flame or charcoal.",
    "Pizza": "A flatbread topped with tomato sauce, melted cheese, and various toppings like meat and vegetables.",
    "Pasta": "Italian noodles served in a creamy sauce with cheese and herbs.",
    "Cold Drink": "Chilled beverages like soft drinks and mineral water.",
}

In [14]:
def restart(user_query):
  if user_query == "":
    print("Input: '' | Error: Empty input.")
    return
  if len(user_query.strip()) < 3:
    print(f"Input: '{user_query}' | Error: Input too short.")
    return
  cat_desc = list(CATEGORY_DESCRIPTIONS.values())
  catch_all = "Irrelevant or non-food request"
  cat_desc.append(catch_all)
  cat_result = classifier(user_query, cat_desc)
  best_cat_desc = cat_result["labels"][0]

  if best_cat_desc == catch_all:
    print(f"Input: '{user_query}' | Error: Not a valid food or drink request.")
    return

  best_category = None
  for cat,desc in CATEGORY_DESCRIPTIONS.items():
    if desc == best_cat_desc:
      best_category = cat
      break

  category_items = []
  for item in FULL_MENU:
    if item["category"] == best_category:
      category_items.append(item)

  item_disc = []
  for item in category_items:
    item_disc.append(item["desc"])

  item_result = classifier(user_query, item_disc)
  best_item_desc = item_result["labels"][0]
  best_item_score = item_result["scores"][0]

  best_item = None
  for item in category_items:
    if item["desc"] == best_item_desc:
      best_item = item
      break
  confidence_percent = round(best_item_score * 100, 2)
  print(f"Category: {best_category} \n Item: {best_item['name']} \n Confidence: {confidence_percent}%\n\n")

test_cravings = [
    "I want a juicy beef burger with extra cheese",
    "I'm craving spicy grilled chicken tikka",
    "I want something crispy and spicy for dinner",
    "I need a new laptop for work",
    "Something quick to eat while walking to work",
    "I want a loaded pizza with lots of meat",
    "I'm craving creamy pasta with cheese",
    "",
    "hi",
    "I want something light and healthy with chicken"
]



for craving in test_cravings:
    restart(craving)


Category: Burger 
 Item: Beef Burger (Cheese) 
 Confidence: 75.3%


Category: BBQ 
 Item: Chicken Bihari Tikka (Leg) 
 Confidence: 60.13%


Category: BBQ 
 Item: Seekh Kabab 
 Confidence: 29.06%


Category: BBQ 
 Item: Seekh Kabab 
 Confidence: 24.34%


Category: Burger 
 Item: Chicken Burger (Cheese) 
 Confidence: 19.13%


Category: Pizza 
 Item: Extra Cheese Topping 
 Confidence: 41.0%


Category: Pasta 
 Item: Passta Large 
 Confidence: 72.9%


Input: '' | Error: Empty input.
Input: 'hi' | Error: Input too short.
Category: Roll 
 Item: Chicken Mayo Roll 
 Confidence: 33.63%


